In [3]:
import faiss
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel, pipeline

# Documents
documents = [
    "Paris is the capital of France.",
    "The Eiffel Tower is in Paris.",
    "Rome is the capital of Italy.",
    "The Colosseum is in Rome.",
    "Tokyo is the capital of Japan.",
    "Mount Fuji is the highest mountain in Japan.",
    "New York is a major city in the USA.",
    "The Statue of Liberty is located in New York.",
    "London is the capital of the UK.",
    "Big Ben is a famous clock tower in London."
]

# HuggingFace BGE Embedding Model
model_name = "BAAI/bge-small-en"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Embedding Function
def embed_text(text):
    encoded = tokenizer(
        text,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        model_output = model(**encoded)
        embeddings = model_output.last_hidden_state.mean(dim=1)

    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    
    return embeddings.cpu().numpy()

# Embeddings & Index Setup
doc_embeddings = embed_text(documents)
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

# Retrieve Function
def retrieve(query, k=3):
    query_vec = embed_text([query])
    distances, indices = index.search(query_vec, k)
    return [documents[i] for i in indices[0]]

#Text Generation Model
generator = pipeline("text2text-generation", model="google/flan-t5-small")

# Query & Retrieval
query = "Where is Mount Fuji"
retrieved_docs = retrieve(query)

print("Retrieved docs:")
for doc in retrieved_docs:
    print("-", doc)

# Context & Prompt
context = " ".join(retrieved_docs)
prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

# Answer
output = generator(prompt, max_new_tokens=100)[0]["generated_text"]

print("\nAnswer:", output)

Device set to use mps:0


Retrieved docs:
- Mount Fuji is the highest mountain in Japan.
- Tokyo is the capital of Japan.
- The Eiffel Tower is in Paris.

Answer: Tokyo
